<a href="https://colab.research.google.com/github/Sougata05/Customer_Support_Chatbot/blob/main/Customer_Support_Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🛍️ ShopEasy Customer Support Chatbot

Run each cell in order. This uses `flask-ngrok` to expose the Flask app from Colab to a public URL.

In [ ]:
# Cell 1: Install dependencies
!pip install -U -q flask anthropic pyngrok

# Force a restart of the internal package metadata if needed
import site
from importlib import reload
reload(site)

<module 'site' (frozen)>

In [ ]:
# Cell 2: Set your API keys
import os
from getpass import getpass

os.environ["ANTHROPIC_API_KEY"] = getpass("Enter your API Key:")

# Get a free ngrok authtoken from https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_AUTH_TOKEN = getpass("Enter your ngrok authtoken: ")

Enter your API Key:··········
Enter your ngrok authtoken: ··········


In [ ]:
import os
os.makedirs("templates", exist_ok=True)

html_content = '''<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>ShopEasy Support</title>
<style>
  @import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600&display=swap');
  *, *::before, *::after { box-sizing: border-box; margin: 0; padding: 0; }
  :root {
    --bg: #0f1117; --surface: #1a1d27; --surface2: #22263a; --border: #2d3148;
    --accent: #6c63ff; --accent2: #a78bfa; --green: #22c55e; --red: #ef4444; --yellow: #f59e0b;
    --text: #e2e8f0; --text-muted: #94a3b8; --user-bubble: #6c63ff; --bot-bubble: #22263a; --radius: 16px;
  }
  body { font-family: 'Inter', sans-serif; background: var(--bg); color: var(--text); height: 100vh; display: flex; align-items: center; justify-content: center; }
  .wrapper { width: 100%; max-width: 480px; height: 100vh; display: flex; flex-direction: column; background: var(--surface); border-left: 1px solid var(--border); border-right: 1px solid var(--border); }
  .header { padding: 16px 20px; background: var(--surface); border-bottom: 1px solid var(--border); display: flex; align-items: center; gap: 12px; }
  .avatar { width: 40px; height: 40px; background: linear-gradient(135deg, var(--accent), var(--accent2)); border-radius: 50%; display: flex; align-items: center; justify-content: center; font-size: 18px; flex-shrink: 0; }
  .header-info h1 { font-size: 15px; font-weight: 600; color: var(--text); }
  .header-info p { font-size: 12px; color: var(--text-muted); display: flex; align-items: center; gap: 5px; margin-top: 2px; }
  .status-dot { width: 7px; height: 7px; background: var(--green); border-radius: 50%; display: inline-block; animation: pulse 2s infinite; }
  @keyframes pulse { 0%, 100% { opacity: 1; } 50% { opacity: 0.5; } }
  .header-actions { margin-left: auto; display: flex; gap: 8px; }
  .icon-btn { background: var(--surface2); border: 1px solid var(--border); color: var(--text-muted); width: 34px; height: 34px; border-radius: 10px; cursor: pointer; display: flex; align-items: center; justify-content: center; font-size: 15px; transition: all 0.2s; }
  .icon-btn:hover { background: var(--border); color: var(--text); }
  .session-bar { padding: 6px 20px; background: var(--surface2); border-bottom: 1px solid var(--border); font-size: 11px; color: var(--text-muted); display: flex; gap: 16px; }
  .session-bar span { display: flex; align-items: center; gap: 4px; }
  .quick-actions { padding: 10px 16px; border-bottom: 1px solid var(--border); display: flex; gap: 6px; overflow-x: auto; scrollbar-width: none; }
  .quick-actions::-webkit-scrollbar { display: none; }
  .chip { background: var(--surface2); border: 1px solid var(--border); color: var(--text-muted); padding: 5px 12px; border-radius: 20px; font-size: 12px; cursor: pointer; white-space: nowrap; transition: all 0.2s; flex-shrink: 0; }
  .chip:hover { border-color: var(--accent); color: var(--accent2); background: rgba(108,99,255,0.1); }
  .messages { flex: 1; overflow-y: auto; padding: 20px 16px; display: flex; flex-direction: column; gap: 14px; scrollbar-width: thin; scrollbar-color: var(--border) transparent; }
  .msg-row { display: flex; align-items: flex-end; gap: 8px; }
  .msg-row.user { flex-direction: row-reverse; }
  .msg-avatar { width: 28px; height: 28px; border-radius: 50%; background: linear-gradient(135deg, var(--accent), var(--accent2)); display: flex; align-items: center; justify-content: center; font-size: 13px; flex-shrink: 0; }
  .msg-row.user .msg-avatar { background: var(--surface2); border: 1px solid var(--border); }
  .bubble { max-width: 78%; padding: 10px 14px; border-radius: var(--radius); font-size: 14px; line-height: 1.55; word-break: break-word; }
  .msg-row.bot .bubble { background: var(--bot-bubble); border: 1px solid var(--border); border-bottom-left-radius: 4px; color: var(--text); }
  .msg-row.user .bubble { background: var(--user-bubble); border-bottom-right-radius: 4px; color: #fff; }
  .time-label { font-size: 10px; color: var(--text-muted); text-align: center; margin: 4px 0; }
  .escalate-banner { background: rgba(245,158,11,0.12); border: 1px solid var(--yellow); border-radius: 10px; padding: 10px 14px; font-size: 13px; color: var(--yellow); display: none; align-items: center; gap: 8px; margin: 4px 0; }
  .escalate-banner.show { display: flex; }
  .typing-indicator { display: none; align-items: center; gap: 8px; padding: 4px 0; }
  .typing-indicator.show { display: flex; }
  .typing-dots { background: var(--bot-bubble); border: 1px solid var(--border); border-radius: var(--radius); border-bottom-left-radius: 4px; padding: 10px 14px; display: flex; gap: 4px; align-items: center; }
  .dot { width: 7px; height: 7px; background: var(--accent2); border-radius: 50%; animation: bounce 1.2s infinite; }
  .dot:nth-child(2) { animation-delay: 0.2s; }
  .dot:nth-child(3) { animation-delay: 0.4s; }
  @keyframes bounce { 0%, 80%, 100% { transform: translateY(0); opacity: 0.4; } 40% { transform: translateY(-6px); opacity: 1; } }
  .sentiment-bar { padding: 6px 16px; background: var(--surface2); border-top: 1px solid var(--border); font-size: 11px; color: var(--text-muted); display: flex; align-items: center; gap: 8px; }
  .sentiment-label { padding: 2px 8px; border-radius: 10px; font-size: 11px; font-weight: 500; }
  .s-neutral { background: rgba(148,163,184,0.15); color: var(--text-muted); }
  .s-positive { background: rgba(34,197,94,0.15); color: var(--green); }
  .s-angry { background: rgba(239,68,68,0.15); color: var(--red); }
  .input-area { padding: 12px 16px; background: var(--surface); border-top: 1px solid var(--border); display: flex; gap: 10px; align-items: flex-end; }
  #userInput { flex: 1; background: var(--surface2); border: 1px solid var(--border); border-radius: 12px; color: var(--text); font-family: 'Inter', sans-serif; font-size: 14px; padding: 10px 14px; resize: none; min-height: 42px; max-height: 120px; outline: none; transition: border-color 0.2s; line-height: 1.4; }
  #userInput:focus { border-color: var(--accent); }
  #userInput::placeholder { color: var(--text-muted); }
  #sendBtn { background: var(--accent); border: none; color: #fff; width: 42px; height: 42px; border-radius: 12px; cursor: pointer; display: flex; align-items: center; justify-content: center; font-size: 18px; flex-shrink: 0; transition: all 0.2s; }
  #sendBtn:hover { background: var(--accent2); transform: scale(1.05); }
  #sendBtn:disabled { opacity: 0.5; cursor: not-allowed; transform: none; }
</style>
</head>
<body>
<div class="wrapper">
  <div class="header">
    <div class="avatar">&#x1f6cd;&#xfe0f;</div>
    <div class="header-info">
      <h1>ShopEasy Support</h1>
      <p><span class="status-dot"></span> Aria &#x00b7; Online</p>
    </div>
    <div class="header-actions">
      <button class="icon-btn" onclick="resetChat()" title="New conversation">&#x21ba;</button>
    </div>
  </div>
  <div class="session-bar">
    <span>&#x1f4cb; Session: <span id="sessionId">&#x2014;</span></span>
    <span>&#x1f4ac; Messages: <span id="msgCount">0</span></span>
  </div>
  <div class="quick-actions">
    <button class="chip" onclick="sendQuick('Track my order')">&#x1f4e6; Track Order</button>
    <button class="chip" onclick="sendQuick('I want to return an item')">&#x21a9;&#xfe0f; Return</button>
    <button class="chip" onclick="sendQuick('Request a refund')">&#x1f4b3; Refund</button>
    <button class="chip" onclick="sendQuick('My payment failed')">&#x274c; Payment Issue</button>
    <button class="chip" onclick="sendQuick('Change delivery address')">&#x1f4cd; Change Address</button>
    <button class="chip" onclick="sendQuick('Cancel my order')">&#x1f6ab; Cancel Order</button>
  </div>
  <div class="messages" id="messages">
    <div class="time-label">Today</div>
    <div class="msg-row bot">
      <div class="msg-avatar">&#x1f916;</div>
      <div class="bubble">Hi there! &#x1f44b; I'm Aria, your ShopEasy support assistant.<br><br>I can help with orders, returns, refunds, payments, and more. What can I do for you today?</div>
    </div>
    <div class="typing-indicator" id="typingIndicator">
      <div class="msg-avatar">&#x1f916;</div>
      <div class="typing-dots"><div class="dot"></div><div class="dot"></div><div class="dot"></div></div>
    </div>
  </div>
  <div class="escalate-banner" id="escalateBanner">
    &#x26a0;&#xfe0f; Your concern has been flagged for a senior specialist. They'll follow up within 2 hours.
  </div>
  <div class="sentiment-bar">
    Tone detected: <span class="sentiment-label s-neutral" id="sentimentLabel">Neutral</span>
  </div>
  <div class="input-area">
    <textarea id="userInput" placeholder="Type your message&#x2026;" rows="1" onkeydown="handleKey(event)" oninput="autoResize(this); detectSentiment(this.value)"></textarea>
    <button id="sendBtn" onclick="sendMessage()">&#x27a4;</button>
  </div>
</div>
<script>
  let currentSentiment = "neutral";
  const angryWords = ["angry","furious","terrible","worst","useless","scam","fraud","disgusting","horrible","ridiculous","pathetic","stupid","idiot","cheated","lied","waste"];
  const positiveWords = ["thanks","great","awesome","perfect","wonderful","excellent","love","happy"];
  function detectSentiment(text) {
    const lower = text.toLowerCase();
    const label = document.getElementById("sentimentLabel");
    if (angryWords.some(w => lower.includes(w))) {
      currentSentiment = "angry"; label.textContent = "Frustrated"; label.className = "sentiment-label s-angry";
    } else if (positiveWords.some(w => lower.includes(w))) {
      currentSentiment = "positive"; label.textContent = "Positive"; label.className = "sentiment-label s-positive";
    } else {
      currentSentiment = "neutral"; label.textContent = "Neutral"; label.className = "sentiment-label s-neutral";
    }
  }
  function autoResize(el) { el.style.height = "auto"; el.style.height = Math.min(el.scrollHeight, 120) + "px"; }
  function handleKey(e) { if (e.key === "Enter" && !e.shiftKey) { e.preventDefault(); sendMessage(); } }
  function sendQuick(text) { document.getElementById("userInput").value = text; sendMessage(); }
  async function sendMessage() {
    const input = document.getElementById("userInput");
    const text = input.value.trim();
    if (!text) return;
    addMessage(text, "user");
    input.value = ""; input.style.height = "auto";
    document.getElementById("sendBtn").disabled = true;
    showTyping(true);
    try {
      const res = await fetch("/chat", { method: "POST", headers: { "Content-Type": "application/json" }, body: JSON.stringify({ message: text, sentiment: currentSentiment }) });
      const data = await res.json();
      showTyping(false);
      if (data.reply) {
        addMessage(data.reply, "bot");
        document.getElementById("sessionId").textContent = data.session_id || "&#x2014;";
        document.getElementById("msgCount").textContent = data.message_count || 0;
        if (data.escalate) document.getElementById("escalateBanner").classList.add("show");
      }
    } catch (err) { showTyping(false); addMessage("Sorry, I'm having trouble connecting. Please try again.", "bot"); }
    document.getElementById("sendBtn").disabled = false;
  }
  function addMessage(text, role) {
    const msgs = document.getElementById("messages");
    const typing = document.getElementById("typingIndicator");
    const row = document.createElement("div");
    row.className = `msg-row ${role}`;
    const avatar = document.createElement("div");
    avatar.className = "msg-avatar";
    avatar.textContent = role === "bot" ? "&#x1f916;" : "&#x1f464;";
    const bubble = document.createElement("div");
    bubble.className = "bubble";
    bubble.innerHTML = text.replace(/\n/g, "<br>");
    row.appendChild(avatar); row.appendChild(bubble);
    msgs.insertBefore(row, typing);
    msgs.scrollTop = msgs.scrollHeight;
  }
  function showTyping(show) { document.getElementById("typingIndicator").classList.toggle("show", show); document.getElementById("messages").scrollTop = 999999; }
  async function resetChat() {
    await fetch("/reset", { method: "POST" });
    const msgs = document.getElementById("messages");
    while (msgs.children.length > 3) msgs.removeChild(msgs.children[1]);
    document.getElementById("escalateBanner").classList.remove("show");
    document.getElementById("sessionId").textContent = "&#x2014;";
    document.getElementById("msgCount").textContent = "0";
    currentSentiment = "neutral";
    document.getElementById("sentimentLabel").textContent = "Neutral";
    document.getElementById("sentimentLabel").className = "sentiment-label s-neutral";
  }
</script>
</body>
</html>
'''

with open("templates/index.html", "w", encoding="utf-8") as f:
    f.write(html_content.encode('utf-8', errors='xmlcharrefreplace').decode('utf-8'))

print("templates/index.html created &#x2705;")

templates/index.html created &#x2705;


In [ ]:
# Cell 4: Flask app
from flask import Flask, render_template, request, jsonify, session
import anthropic
import uuid
from datetime import datetime

app = Flask(__name__)
app.secret_key = "support_chatbot_secret_key_2024"

client = anthropic.Anthropic()

SYSTEM_PROMPT = """You are a friendly and empathetic customer support agent for ShopEasy, an e-commerce platform. Your name is Aria.

Your capabilities:
- Handle order tracking, cancellations, and modifications
- Process return and refund requests
- Troubleshoot payment issues
- Answer product questions and availability
- Resolve complaints with empathy and professionalism

Guidelines:
- Always greet the customer warmly on first message
- Show genuine empathy for frustrated customers
- Be concise but thorough — no unnecessary filler
- For order issues, ask for order ID if not provided
- For refunds, confirm item, reason, and preferred refund method
- If a problem is complex or requires actual system access, say:
  "I'm flagging this for our specialist team — they'll reach out within 2 hours."
- Never make up order statuses or policies you're unsure about
- End conversations by asking if there's anything else you can help with

Tone: Professional, warm, solution-focused. Not robotic.

Common policies to know:
- Returns accepted within 30 days of delivery
- Refunds processed in 5-7 business days
- Free shipping on orders over
499
- Customer support hours: 9 AM – 9 PM IST
"""

def get_session_history():
    if "history" not in session:
        session["history"] = []
        session["session_id"] = str(uuid.uuid4())[:8].upper()
        session["start_time"] = datetime.now().isoformat()
    return session["history"]

@app.route("/")
def index():
    session.clear()
    return render_template("index.html")

@app.route("/chat", methods=["POST"])
def chat():
    data = request.json
    user_message = data.get("message", "").strip()
    sentiment = data.get("sentiment", "neutral")

    if not user_message:
        return jsonify({"error": "Empty message"}), 400

    history = get_session_history()

    system = SYSTEM_PROMPT
    if sentiment == "angry":
        system += "\n\nIMPORTANT: This customer appears frustrated. Lead with empathy, acknowledge their frustration explicitly before offering solutions."

    history.append({"role": "user", "content": user_message})

    response = client.messages.create(
        model="claude-3-sonnet-20240229",
        max_tokens=1000,
        system=system,
        messages=history
    )

    assistant_message = response.content[0].text
    history.append({"role": "assistant", "content": assistant_message})
    session["history"] = history
    session.modified = True

    escalate = any(word in user_message.lower() for word in [
        "manager", "lawsuit", "legal", "fraud", "scam", "worst", "terrible", "horrible"
    ])

    return jsonify({
        "reply": assistant_message,
        "escalate": escalate,
        "session_id": session.get("session_id", "N/A"),
        "message_count": len(history) // 2
    })

@app.route("/reset", methods=["POST"])
def reset():
    session.clear()
    return jsonify({"status": "ok"})

print("Flask app defined ✅")

Flask app defined ✅


In [ ]:
# Cell 5: Launch with ngrok tunnel
from pyngrok import ngrok, conf
import threading
import os
import sys

# Kill any process currently using port 5000
!fuser -k 5000/tcp

# Ensure NGROK_AUTH_TOKEN and Flask app exist
if 'app' not in globals():
    print("❌ Error: 'app' is not defined. Please run Cell 4 (Flask app definition) first!")
else:
    try:
        conf.get_default().auth_token = NGROK_AUTH_TOKEN
    except NameError:
        print("❌ Error: NGROK_AUTH_TOKEN is not defined. Please run Cell 2 first!")
        raise

    # Kill any existing ngrok tunnels
    ngrok.kill()

    public_url = ngrok.connect(5000)
    print(f"🚀 Your chatbot is live at: {public_url}\n")

    def run_app():
        # Use use_reloader=False to prevent issues in Colab threads
        app.run(port=5000, use_reloader=False)

    thread = threading.Thread(target=run_app)
    thread.daemon = True
    thread.start()

    print("Click the URL above to open your chatbot 👆")

🚀 Your chatbot is live at: NgrokTunnel: "https://acclaim-supply-browse.ngrok-free.dev" -> "http://localhost:5000"

Click the URL above to open your chatbot 👆
